# M3 baselines (B0 to B5), the C1 few-shot curve and K1 enrolment

We score each baseline on our validation photos before any training on them, so every trained model has a floor and
a reference to beat. The same notebook runs C1, the few-shot curve: prototypes from 5, 10 and 20 photos of distinct
observations on tier 2 classes, on DINOv2 features and, once an E2 run exists, on E2 features. K1 enrols the K1 run's
ten held-out species from 1, 5 and 10 photos.

- **B0** predicts the class frequencies of the capped subset.
- **B1** is Bolon et al. 2022's global ViT (ONNX from LINDAT), with its labels mapped to our classes. On full
  validation it is an upper bound (`upper_bound_overlap_unverified`); the fair score is on validation photos uploaded
  from 2022 (`upload_2022_slice`), after its training data ends.
- **B2** is BioCLIP zero-shot from taxonomic prompts. Its training data runs past 2022, so even the 2022 slice is
  "overlap not verified".
- **B3** is each of the three E1 encoders, frozen with ImageNet weights, scored by the nearest prototype from the
  subset.
- **B4** is DINOv2 ViT-S/14 frozen at 322 px, with prototypes and a linear probe.
- **B5** is YOLOv8n trained on Open Images V7, used as is on the Open Images snake boxes.
- **B6** is not run; the note after the B5 cell says why.

**Inputs:** `serpentguard-indian-snakes-clean`, `serpentguard-sam-masks`, `serpentguard-m3-subset` and
`serpentguard-extended-sources`, found with `kagglehub.dataset_download`. The baselines' code is in this notebook; the shared code comes from `../src` and
`../configs/m3`, which the runner kernel unpacks into `/kaggle/working/repo`.

**Outputs:** each baseline writes `/kaggle/working/baselines/<B-id>/final_metrics.json` in the training runner's shape
(head, safety, calibration, taxonomy, run) with its score on the 2022 slice, `versions.json`, and `predictions.npz`
where there are per-photo predictions. The last cell writes `comparison.csv` and prints it. C1 and K1 write to
`/kaggle/working/fewshot/`.

**Run:** on a Kaggle GPU with internet on (pip, LINDAT, Hugging Face and Ultralytics weights). `RUN_BASELINES` picks
the baselines and `LIMIT` makes a smoke run; `E2_RUN_DIR` and `K1_RUN_DIR` turn on C1 on E2 features and K1.

In [1]:
RUN_BASELINES = ["B0", "B1", "B2", "B3", "B4", "C1", "B5"]  # drop an id to skip it
LIMIT = None       # first N photos of each frame (and N Open Images photos), for a smoke run
E2_RUN_DIR = None  # an E2 run's output folder (val_outputs.npz, calib_outputs.npz): C1 on E2 features
K1_RUN_DIR = None  # the K1 run's output folder (heldout_outputs.npz, prototypes.npz, val_outputs.npz)

In [2]:
import os
import subprocess
import sys

ALBUMENTATIONS_VERSION = "2.0.8"  # requirements-m3.txt; the per-transform augmentation seeding was verified on it
TIMM_VERSION = "1.0.31"            # requirements-m3.txt; the encoders' feature maps were measured on it
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"albumentations=={ALBUMENTATIONS_VERSION}",
                f"timm=={TIMM_VERSION}"], check=True)
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"  # otherwise albumentations asks PyPI for its latest version on import

import albumentations
import timm

assert albumentations.__version__ == ALBUMENTATIONS_VERSION, f"albumentations {albumentations.__version__} is loaded"
assert timm.__version__ == TIMM_VERSION, f"timm {timm.__version__} is loaded"
print(f"albumentations {albumentations.__version__}, timm {timm.__version__}")

albumentations 2.0.8, timm 1.0.31


In [3]:
SCIKIT_LEARN_VERSION = "1.9.1"  # requirements-m3.txt; B4's probe and every metric use it
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"scikit-learn=={SCIKIT_LEARN_VERSION}"], check=True)

import sklearn

assert sklearn.__version__ == SCIKIT_LEARN_VERSION, (f"scikit-learn {sklearn.__version__} is loaded; restart the "
                                                     f"kernel after the install")
print(f"scikit-learn {sklearn.__version__}")

scikit-learn 1.9.1


In [4]:
OPEN_CLIP_VERSION = "2.32.0"  # BioCLIP (B2) loads through open_clip; timm stays at the pin above
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"open_clip_torch=={OPEN_CLIP_VERSION}",
                f"timm=={TIMM_VERSION}"], check=True)

import open_clip

assert open_clip.__version__ == OPEN_CLIP_VERSION, f"open_clip {open_clip.__version__} is loaded"
EXTRA_VERSIONS = {"open_clip_torch": open_clip.__version__}
print(f"open_clip {open_clip.__version__}")

open_clip 2.32.0


In [5]:
ONNXRUNTIME_VERSION = "1.31.1"  # requirements-m3.txt's onnxruntime, as the GPU build: B1's ViT is 1.13 GB
ONNX_VERSION = "1.23.2"         # requirements-m3.txt; reads B1's graph to show how it preprocesses
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y", "onnxruntime"], check=True)  # one build only
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"onnxruntime-gpu=={ONNXRUNTIME_VERSION}",
                f"onnx=={ONNX_VERSION}"], check=True)

import onnx
import onnxruntime

assert onnxruntime.__version__ == ONNXRUNTIME_VERSION, f"onnxruntime {onnxruntime.__version__} is loaded"
assert onnx.__version__ == ONNX_VERSION, f"onnx {onnx.__version__} is loaded"
EXTRA_VERSIONS.update({"onnxruntime-gpu": onnxruntime.__version__, "onnx": onnx.__version__})
print(f"onnxruntime {onnxruntime.__version__}, providers {onnxruntime.get_available_providers()}")

onnxruntime 1.31.1, providers ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


In [6]:
ULTRALYTICS_VERSION = "8.4.202"  # requirements-m3.txt; B5 runs yolov8n-oiv7.pt through it
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"ultralytics=={ULTRALYTICS_VERSION}"], check=True)

import ultralytics

assert ultralytics.__version__ == ULTRALYTICS_VERSION, f"ultralytics {ultralytics.__version__} is loaded"
EXTRA_VERSIONS["ultralytics"] = ultralytics.__version__
print(f"ultralytics {ultralytics.__version__}")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
ultralytics 8.4.202


In [7]:
import hashlib
import json
import shutil
import time
import urllib.request
from pathlib import Path

import kagglehub

CODE = Path("/kaggle/input/serpentguard-code/src")
CODE = CODE if CODE.exists() else Path("../src").resolve()
sys.path.insert(0, str(CODE))
CONFIGS = CODE.parent / "configs" / "m3"

import numpy as np
import pandas as pd
import torch

from serpentguard.frontend_scores import (YOLO_ALL_CONF, YOLO_BATCH, YOLO_CONF, YOLO_WEIGHTS,
                                          detection_baseline_scores, open_images_ground_truth, snake_class_index,
                                          yolo_snake_boxes)
from serpentguard.config import load_config
from serpentguard.metrics import DET_IOU
from serpentguard.models import DINOV2, INPUT_SIZE, MOBILE_ENCODERS
from serpentguard.run_data import read_tables, run_frames
from serpentguard.run_outputs import class_tables, subset_hash, write_json

## The baselines' code

The first cell scores a baseline (full validation and the 2022 upload slice) and saves it, as every baseline below
does; the second holds B1's label mapping and ONNX feed, B2's prompts and B3 and B4's frozen encoders. B5's box
finder is the app's, in `serpentguard.frontend_scores`.

In [8]:
import math
from pathlib import Path

import numpy as np

from serpentguard.metrics import calibration as calibration_scores, safety_report, species_report, taxonomy_report
from serpentguard.run_outputs import library_versions, save_outputs, write_json
from serpentguard.train import FINAL_FILE, VERSIONS_FILE

FULL_VALIDATION = "full_validation"
UPPER_BOUND = "upper_bound_overlap_unverified"   # B1, B2 on full validation: their training photos may overlap it
UPLOAD_2022 = "upload_2022_slice"
SLICE_UPLOADED_FROM = "2022-01-01"   # notebook 02's TEST_CREATED_FROM: created_at compared as text, local time
NO_OVERLAP = "no labelled photos from outside our training subset"
OVERLAP_CLEARED = "clear by date: the model's training photos predate 2022"
OVERLAP_NOT_VERIFIED = "overlap not verified: the model's training data runs past 2022"
PATH, CLASS_ID = "path", "class_id"


def prior_baseline(train_counts, n_val):
    """B0: every photo gets the training class frequencies."""
    counts = np.asarray(train_counts, dtype=float)
    return np.tile(counts / counts.sum(), (n_val, 1))


def upload_2022_slice(frame, metadata, uploaded_from=SLICE_UPLOADED_FROM):
    """Per frame row: whether its photo was uploaded on or after uploaded_from (metadata created_at)."""
    created = frame[[PATH]].merge(metadata[[PATH, "created_at"]], on=PATH, how="left", indicator=True,
                                  validate="many_to_one")
    missing = created["_merge"].eq("left_only")
    if missing.any():
        raise ValueError(f"{int(missing.sum())} photos are missing from metadata, e.g. "
                         f"{created.loc[missing, PATH].iloc[0]}")
    return created["created_at"].fillna("").astype(str).ge(uploaded_from).to_numpy()


def baseline_metrics(val, probs, train_counts, taxa, in_slice, run, evaluation=FULL_VALIDATION,
                     slice_overlap=NO_OVERLAP, prototype_probs=None, extra=None):
    """final_metrics.json for a classification baseline, on full validation and on the 2022 upload slice, which keeps
    full validation's measurable classes."""
    y_true = val[CLASS_ID].to_numpy()
    val_counts = np.bincount(y_true, minlength=taxa.n_classes)
    genus, family = val["genus_id"].to_numpy(), val["family_id"].to_numpy()
    in_slice = np.asarray(in_slice, dtype=bool)

    def scored(rows):
        section = {"head": species_report(y_true[rows], probs[rows], train_counts, val_counts, taxa.tier)}
        if prototype_probs is not None:
            section["prototype"] = species_report(y_true[rows], prototype_probs[rows], train_counts, val_counts,
                                                  taxa.tier)
        return {**section, "safety": safety_report(y_true[rows], probs[rows], taxa.who, taxa.listed_or_venomous),
                "calibration": calibration_scores(probs[rows], y_true[rows]),
                "taxonomy": taxonomy_report(probs[rows], None, None, taxa.genus_of_class, taxa.family_of_class,
                                            genus[rows], family[rows])}

    return {**scored(np.ones(len(y_true), dtype=bool)),
            UPLOAD_2022: {"label": UPLOAD_2022, "overlap": slice_overlap, "photos": int(in_slice.sum()),
                          "classes": int(np.unique(y_true[in_slice]).size), **scored(in_slice)},
            **(extra or {}), "run": {**run, "evaluation": evaluation, "val_photos": len(y_true)}}


def save_baseline(out_dir, metrics, extra_versions=None, frame=None, outputs=None):
    """Writes versions.json, final_metrics.json and (given a frame) predictions.npz; returns the metrics read back."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    write_json(out_dir / VERSIONS_FILE, {**library_versions(), **(extra_versions or {})})
    if frame is not None:
        save_outputs(out_dir / "predictions.npz", frame, outputs)
    return write_json(out_dir / FINAL_FILE, metrics)


def summary_row(name, final):
    """One comparison row from a baseline's or a trained run's final_metrics.json; absent numbers are nan."""
    def pick(*keys):
        value = final
        for key in keys:
            if not isinstance(value, dict) or key not in value:
                return math.nan
            value = value[key]
        return value

    return {"model": name, "evaluation": final.get("run", {}).get("evaluation", FULL_VALIDATION),
            "macro_f1": pick("head", "macro_f1"), "macro_f1_prototype": pick("prototype", "macro_f1"),
            "top1": pick("head", "top1"), "top3": pick("head", "top3"),
            "who_recall": pick("safety", "who_recall_raw"), "o3_raw": pick("safety", "top1_error_raw"),
            "ece": pick("calibration", "ece"), "macro_f1_2022": pick(UPLOAD_2022, "head", "macro_f1"),
            "top1_2022": pick(UPLOAD_2022, "head", "top1"), "slice_overlap": pick(UPLOAD_2022, "overlap"),
            "class_coverage": pick("coverage", "class_share"), "photo_coverage": pick("coverage", "photo_share"),
            "precision": pick("detection", "precision"), "recall": pick("detection", "recall"),
            "map50": pick("detection", "map50")}

In [9]:
import time
from types import SimpleNamespace

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset

from serpentguard.data import SnakeDataset, load_rgb
from serpentguard.fewshot import cosine_probs
from serpentguard.models import ARCFACE_S, INPUT_SIZE, build_encoder, prototypes_from
from serpentguard.run_outputs import softmax
from serpentguard.train import encoder_normalization

UNMAPPED = -1
SYNONYMS = {  # older names a published model may use -> our classes.csv name (M1 report, section 3)
    "Xenochrophis piscator": "Fowlea piscator",
    **{f"Trimeresurus {epithet}": f"Craspedocephalus {epithet}"
       for epithet in ("malabaricus", "gramineus", "anamallensis", "travancoricus", "macrolepis")},
    "Macropisthodon plumbicolor": "Rhabdophis plumbicolor",
    "Ramphotyphlops braminus": "Indotyphlops braminus",
}
LABEL_ORDER_FLOOR = 0.2              # B1's check top-1 must reach this, or 10 x chance if that is lower
BOLON_NAME_COLUMN, BOLON_ID_COLUMN = "binomial", "class_id"   # SnakeCLEF 2021 metadata
ONNX_PROVIDERS = ("CUDAExecutionProvider", "CPUExecutionProvider")
ONNX_DTYPES = {"tensor(uint8)": np.uint8, "tensor(int32)": np.int32, "tensor(int64)": np.int64,
               "tensor(float)": np.float32, "tensor(float16)": np.float16}
GRAPH_PREVIEW_NODES = 20
SMALL_CONSTANT_SIZE = 16             # a constant this small may be a mean, a std or a resize target
BIOCLIP_HUB = "hf-hub:imageomics/bioclip"
BIOCLIP_LINEAGE = "Animalia Chordata Reptilia Squamata"
BIOCLIP_TEMPLATES = ("a photo of {}.",)
EMBED_BATCH = 64
DINOV2_INPUT_SIZE = 322              # 23 x 23 patches of 14
PROTOTYPE_SCALE = ARCFACE_S          # the runner scores prototypes as softmax(arcface_s x cosine)
PROBE_C, PROBE_MAX_ITER = 1.0, 1000
OPEN_IMAGES_EVALUATION = "open_images_snake_boxes"


def normalise_name(name):
    return " ".join(str(name).replace("_", " ").split()).lower()


def map_labels(their_names, our_classes, synonyms=SYNONYMS):
    """(index_map, coverage): our class id for each of their labels, or UNMAPPED; names match ignoring case,
    underscores and spacing, or through synonyms. A None label stays unmapped."""
    ours = {normalise_name(name): class_id for class_id, name in enumerate(our_classes)}
    renames = {normalise_name(old): normalise_name(new) for old, new in synonyms.items()}
    index_map, renamed = np.full(len(their_names), UNMAPPED, dtype=int), {}
    for index, name in enumerate(their_names):
        if name is None:
            continue
        key = normalise_name(name)
        if key in ours:
            index_map[index] = ours[key]
        elif renames.get(key) in ours:
            index_map[index] = ours[renames[key]]
            renamed[str(name)] = our_classes[index_map[index]]
    covered = sorted(set(index_map[index_map != UNMAPPED].tolist()))
    return index_map, {
        "classes": len(our_classes), "classes_covered": len(covered), "class_share": len(covered) / len(our_classes),
        "covered_class_ids": covered,
        "uncovered_species": [name for class_id, name in enumerate(our_classes) if class_id not in set(covered)],
        "their_labels": len(their_names), "their_labels_mapped": int((index_map != UNMAPPED).sum()),
        "their_labels_unnamed": sum(name is None for name in their_names), "renamed": renamed}


def remap_probs(their_probs, index_map, n_classes):
    """Their probabilities moved to our classes and renormalised; a photo with no mapped mass gets a uniform spread
    over the covered classes."""
    their_probs, index_map = np.asarray(their_probs, dtype=float), np.asarray(index_map)
    mapped = index_map != UNMAPPED
    probs = their_probs[:, mapped] @ np.eye(n_classes)[index_map[mapped]]
    covered = np.zeros(n_classes)
    covered[np.unique(index_map[mapped])] = 1
    probs[probs.sum(axis=1) <= 0] = covered
    return probs / probs.sum(axis=1, keepdims=True)


def photo_share(labels, covered_class_ids):
    return float(np.isin(np.asarray(labels), covered_class_ids).mean())


def check_label_order(probs, labels, covered_class_ids, floor=LABEL_ORDER_FLOOR, chance_multiple=10):
    """B1's guard: top-1 on covered classes must reach min(floor, chance_multiple x chance), or a label list in the
    wrong order, which scores about chance, would be written as a result."""
    labels = np.asarray(labels)
    top1 = float((np.asarray(probs).argmax(axis=1) == labels).mean())
    chance = 1 / len(covered_class_ids)
    threshold = min(floor, chance_multiple * chance)
    if top1 < threshold:
        raise ValueError(f"B1 top-1 {top1:.3f} on {len(labels)} photos of covered classes is below {threshold:.3f} "
                         f"(chance {chance:.4f}). Check the label order (output index ≠ class_id) first, then the "
                         f"input preparation")
    return {"top1": top1, "threshold": threshold, "chance": chance, "photos": len(labels)}


def labels_from_metadata(table, n_outputs, name_column=BOLON_NAME_COLUMN, id_column=BOLON_ID_COLUMN):
    """A model's n_outputs labels from its training metadata (one row per photo), None for an id with no row."""
    pairs = table[[id_column, name_column]].dropna().drop_duplicates()
    names_per_id = pairs.groupby(id_column)[name_column].nunique()
    if (names_per_id > 1).any():
        raise ValueError(f"class ids {names_per_id[names_per_id > 1].index.tolist()[:5]} have more than one name")
    ids = pairs[id_column].astype(int).to_numpy()
    outside = sorted(set(ids[(ids < 0) | (ids >= n_outputs)].tolist()))
    if outside:
        raise ValueError(f"{id_column} {outside[:5]} lies outside the model's {n_outputs} outputs")
    labels = [None] * n_outputs
    for class_id, name in zip(ids, pairs[name_column].astype(str)):
        labels[class_id] = name
    return labels


class ImageFiles(Dataset):
    """Photos as stored (RGB, no exposure fix), each passed through transform."""

    def __init__(self, paths, transform):
        self.paths, self.transform = [str(path) for path in paths], transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with Image.open(self.paths[index]) as image:
            return self.transform(image.convert("RGB"))


class LoadedPhotos(ImageFiles):
    """Photos read as the runner reads them (load_rgb, with the exposure fix), passed through transform."""

    def __getitem__(self, index):
        return self.transform(load_rgb(self.paths[index]))


class OnnxClassifier:
    """An ONNX classifier that sizes and normalises photos itself, like B1's graph: its [H, W, 3] input takes one
    load_rgb photo per run at the photo's own size. input_size, mean and std are for graphs that don't, which none of
    our runs met, so they must stay None; batch_size is ignored."""

    def __init__(self, onnx_path, labels, input_size=None, mean=None, std=None, providers=ONNX_PROVIDERS,
                 session=None, channel_order="RGB"):
        if input_size is not None or mean is not None or std is not None:
            raise ValueError("only a graph that sizes and normalises photos itself is fed here")
        self.session = session or open_onnx(onnx_path, providers)
        spec, output = self.session.get_inputs()[0], self.session.get_outputs()[0]
        if len(spec.shape) != 3 or spec.shape[2] != 3 or spec.shape[0] == 3:
            raise ValueError(f"can't tell the channel axis of input shape {spec.shape}: it must be [H, W, 3]")
        if isinstance(output.shape[-1], int) and output.shape[-1] != len(labels):
            raise ValueError(f"the model has {output.shape[-1]} outputs but {len(labels)} labels were given")
        self.input_name, self.input_type, self.input_dtype = spec.name, spec.type, ONNX_DTYPES[spec.type]
        self.output_name, self.labels, self.channel_order = output.name, list(labels), channel_order

    def describe(self):
        """How photos are fed to the model."""
        return {"input": self.input_name, "output": self.output_name, "input_size": None, "layout": "HWC",
                "dtype": self.input_type, "fixed_batch": None,
                "feed": "one photo per run at its own size, after load_rgb", "channel_order": self.channel_order,
                "mean": None, "std": None, "labels": len(self.labels), "providers": self.session.get_providers()}

    def image_array(self, rgb):
        pixels = np.asarray(rgb, dtype=np.float32)
        if self.channel_order == "BGR":
            pixels = pixels[..., ::-1]
        return np.ascontiguousarray(pixels.astype(self.input_dtype))

    def predict_files(self, paths, batch_size=EMBED_BATCH, num_workers=0, progress_every=None):
        """Probabilities for every photo, in order; progress_every prints a line every that many photos."""
        loader = DataLoader(LoadedPhotos(paths, self.image_array), batch_size=None, num_workers=num_workers)
        scores, started = [], time.perf_counter()
        for item in loader:
            raw = self.session.run([self.output_name], {self.input_name: item.numpy()})[0]
            scores.append(np.asarray(raw).reshape(-1, len(self.labels)))
            if progress_every and (len(scores) % progress_every == 0 or len(scores) == len(paths)):
                print(f"{len(scores)}/{len(paths)} photos, {len(scores) / (time.perf_counter() - started):.1f} per "
                      f"second", flush=True)
        scores = np.concatenate(scores).astype(np.float64)
        is_probability = (scores >= 0).all() and np.allclose(scores.sum(axis=1), 1, atol=1e-3)
        return scores if is_probability else softmax(scores)


def open_onnx(onnx_path, providers=ONNX_PROVIDERS):
    """An onnxruntime session on the providers this build has (CUDA, else CPU), with telemetry off."""
    import onnxruntime

    onnxruntime.disable_telemetry_events()
    available = onnxruntime.get_available_providers()
    return onnxruntime.InferenceSession(str(onnx_path), providers=[name for name in providers if name in available])


def describe_onnx_graph(onnx_path, first_nodes=GRAPH_PREVIEW_NODES, max_constant=SMALL_CONSTANT_SIZE):
    """What an ONNX graph does before the network: its inputs and outputs, first nodes, the ops before the first
    convolution (a Resize there means it sizes photos itself) and the small constants they read."""
    import onnx
    from onnx import TensorProto, numpy_helper

    def info(value):
        tensor_type = value.type.tensor_type
        return {"name": value.name, "type": TensorProto.DataType.Name(tensor_type.elem_type).lower(),
                "shape": [dim.dim_param or dim.dim_value for dim in tensor_type.shape.dim]}

    def plain(value):
        if isinstance(value, bytes):
            return value.decode(errors="replace")
        if isinstance(value, (list, tuple)):
            return [plain(item) for item in value]
        return value if isinstance(value, (int, float, str)) else type(value).__name__

    graph = onnx.load(str(onnx_path)).graph
    tensors = {tensor.name: tensor for tensor in graph.initializer}
    tensors.update({node.output[0]: attribute.t for node in graph.node if node.op_type == "Constant"
                    for attribute in node.attribute if attribute.name == "value"})
    ops = [node.op_type for node in graph.node]
    first_conv = next((index for index, op in enumerate(ops) if op == "Conv"), None)
    before_conv = ops[:len(ops) if first_conv is None else first_conv]
    small = {}
    for node in graph.node[:max(first_nodes, first_conv or 0)]:
        for name in node.input:
            if name in tensors and name not in small:
                values = numpy_helper.to_array(tensors[name])
                if values.size <= max_constant:
                    small[name] = values.ravel().tolist()
    return {"inputs": [info(value) for value in graph.input if value.name not in tensors],
            "outputs": [info(value) for value in graph.output],
            "nodes": len(ops), "ops_before_first_conv": before_conv,
            "resizes": any(op in ("Resize", "Upsample") for op in before_conv),
            "first_nodes": [{"op": node.op_type, "inputs": list(node.input), "outputs": list(node.output),
                             "attributes": {attribute.name: plain(onnx.helper.get_attribute_value(attribute))
                                            for attribute in node.attribute if attribute.name != "value"}}
                            for node in graph.node[:first_nodes]],
            "small_constants": small}


def bolon_baseline(classifier, frame, our_classes, synonyms=SYNONYMS, batch_size=EMBED_BATCH, num_workers=0,
                   progress_every=None):
    """B1: (probs over our classes, coverage); classes it can't predict get 0, so their photos are misses."""
    index_map, coverage = map_labels(classifier.labels, list(our_classes), synonyms)
    their_probs = classifier.predict_files(frame["image_file"].tolist(), batch_size=batch_size,
                                           num_workers=num_workers, progress_every=progress_every)
    probs = remap_probs(their_probs, index_map, len(our_classes))
    coverage.update(photos=len(frame), photo_share=photo_share(frame[CLASS_ID], coverage["covered_class_ids"]))
    return probs, coverage


def taxonomic_names(classes):
    """BioCLIP's taxonomic string per class, kingdom to epithet, in class_id order."""
    names = []
    for row in classes.sort_values(CLASS_ID).itertuples():
        genus, epithet = row.species.split(" ", 1)
        if genus != row.genus:
            raise ValueError(f"{row.species!r} has genus {row.genus!r} in classes.csv")
        names.append(f"{BIOCLIP_LINEAGE} {row.family} {row.genus} {epithet}")
    return names


def logit_scale(model):
    """A CLIP model's logit scale: open_clip stores its log as `logit_scale`."""
    return float(torch.as_tensor(model.logit_scale).detach().double().exp())


@torch.inference_mode()
def bioclip_baseline(model, tokenizer, preprocess, frame, class_names, templates=BIOCLIP_TEMPLATES,
                     batch_size=EMBED_BATCH, num_workers=0, device="cpu"):
    """B2: softmax(logit scale x cosine) between each photo and each class's mean text embedding over templates."""
    device = torch.device(device)
    model.eval()
    prompts = [template.format(name) for name in class_names for template in templates]
    with torch.autocast(device.type, enabled=device.type == "cuda"):
        text = F.normalize(model.encode_text(tokenizer(prompts).to(device)).float(), dim=-1)
        text = F.normalize(text.reshape(len(class_names), len(templates), -1).mean(dim=1), dim=-1)
        loader = DataLoader(ImageFiles(frame["image_file"], preprocess), batch_size=batch_size,
                            num_workers=num_workers)
        images = torch.cat([F.normalize(model.encode_image(batch.to(device)).float(), dim=-1) for batch in loader])
    return softmax(logit_scale(model) * (images @ text.T).cpu().double().numpy())


def frozen_encoder(encoder_name, input_size=INPUT_SIZE, pretrained=True):
    """build_encoder's module, input adapter included, for B3 and B4."""
    return build_encoder(encoder_name, pretrained=pretrained, input_size=input_size)[0]


def frozen_run_record(encoder_name, encoder, input_size):
    return {"encoder": encoder_name, "input_size": input_size, "prototype_scale": PROTOTYPE_SCALE,
            "encoder_normalization": encoder_normalization(SimpleNamespace(encoder=encoder))}


@torch.inference_mode()
def embed_frozen(encoder_name, frame, input_size=INPUT_SIZE, batch_size=EMBED_BATCH, num_workers=0, device=None,
                 pretrained=True, encoder=None):
    """(N, C) float32 global-average-pooled final feature map per photo, letterboxed whole as the runner reads it."""
    device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
    if encoder is None:
        encoder, _ = build_encoder(encoder_name, pretrained=pretrained, input_size=input_size)
    encoder = encoder.to(device).eval()
    dataset = SnakeDataset(frame.assign(mask_file=""), train=False, size=input_size)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers,
                        pin_memory=device.type == "cuda")
    pooled = []
    for batch in loader:
        with torch.autocast(device.type, enabled=device.type == "cuda"):
            pooled.append(encoder(batch["image"].to(device, non_blocking=True))[-1].float().mean((2, 3)))
    return torch.cat(pooled).cpu().numpy().astype(np.float32)


def frozen_prototype_baseline(features_train, y_train, features_val, scale=PROTOTYPE_SCALE, n_classes=None):
    """B3, B4: softmax(scale x cosine) to each class's mean normalised training feature; a class without one gets 0."""
    y_train = np.asarray(y_train, dtype=np.int64)
    n_classes = n_classes or int(y_train.max()) + 1
    prototypes = prototypes_from(torch.as_tensor(np.asarray(features_train), dtype=torch.float32), y_train, n_classes)
    return cosine_probs(features_val, prototypes.numpy(), scale)


def linear_probe(features_train, y_train, features_val, n_classes=None, c=PROBE_C, max_iter=PROBE_MAX_ITER):
    """B4: balanced logistic regression on L2-normalised features; classes absent from training get 0."""
    from sklearn.linear_model import LogisticRegression

    def unit_rows(features):
        features = np.asarray(features, dtype=float)
        return features / np.linalg.norm(features, axis=1, keepdims=True)

    y_train = np.asarray(y_train, dtype=int)
    probe = LogisticRegression(C=c, max_iter=max_iter, class_weight="balanced").fit(unit_rows(features_train), y_train)
    probs = np.zeros((len(features_val), n_classes or int(y_train.max()) + 1))
    probs[:, probe.classes_] = probe.predict_proba(unit_rows(features_val))
    return probs

In [10]:
OWNER = "puneetbajajiitm"


def input_root(slug):
    """Where an attached dataset is mounted; Kaggle no longer mounts datasets at /kaggle/input/<slug>."""
    return Path(kagglehub.dataset_download(f"{OWNER}/{slug}"))


DATA = input_root("serpentguard-indian-snakes-clean")
MASKS = input_root("serpentguard-sam-masks")
SUBSET = input_root("serpentguard-m3-subset") / "subset.csv"
EXTENDED = input_root("serpentguard-extended-sources")
OUT = Path("/kaggle/working/baselines")
FEWSHOT_OUT = Path("/kaggle/working/fewshot")
DOWNLOADS = Path("/kaggle/tmp/baselines")  # outside /kaggle/working, so the 1.13 GB model isn't saved as output
FRAME_CONFIG = "E1_mnv4"  # F0, whole photo: the runner's capped subset and full validation, as every E1 run sees them
ENCODER_IDS = {"mobilenetv4_conv_small": "mnv4", "efficientnet_b0": "effb0", "mobilevit_s": "mvits"}  # E1's suffixes
EXPECTED_SUBSET_HASH = "40196d19a68e"  # first 12 hex digits of the real subset's SHA-256
EXPECTED_SLICE_PHOTOS = 13_947         # validation photos uploaded from 2022
TIER2 = "tier2"
BATCH = 64
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WORKERS = os.cpu_count() or 2

for required in (DATA / "metadata.csv", DATA / "classes.csv", DATA / "data", SUBSET,
                 EXTENDED / "open_images_snake_boxes.csv", EXTENDED / "open_images_snake_masks.csv"):
    assert required.exists(), f"missing input {required}"
assert set(ENCODER_IDS) == set(MOBILE_ENCODERS), "B3 must cover exactly the three E1 encoders"

cfg = load_config(CONFIGS / f"{FRAME_CONFIG}.yaml")
tables = read_tables(cfg, DATA, MASKS, SUBSET)
frames = run_frames(cfg, tables, DATA, MASKS, limit=LIMIT)
taxa = class_tables(tables.classes)
train, val, calibration = frames.train, frames.val, frames.calibration
train_counts = np.bincount(train["class_id"], minlength=taxa.n_classes)
in_2022 = upload_2022_slice(val, tables.metadata)
digest = subset_hash(tables.subset["path"])
assert digest.startswith(EXPECTED_SUBSET_HASH), f"another subset was read: {digest}"
if LIMIT is None:
    assert in_2022.sum() == EXPECTED_SLICE_PHOTOS, f"{in_2022.sum()} validation photos are uploaded from 2022"
RUN = {"subset_hash": digest, "train_photos": len(train), "device": str(DEVICE), "limit": LIMIT,
       "config": {"frame_config": FRAME_CONFIG, "front_end": cfg.front_end, "frame_input_size": cfg.input_size,
                  "slice_uploaded_from": SLICE_UPLOADED_FROM, "batch": BATCH, "workers": WORKERS,
                  "subset": str(SUBSET), "expected_subset_hash": EXPECTED_SUBSET_HASH}}
OUT.mkdir(parents=True, exist_ok=True)
FEWSHOT_OUT.mkdir(parents=True, exist_ok=True)
print(f"code {CODE}; {len(train)} training photos after caps, {len(val)} validation ({int(in_2022.sum())} uploaded "
      f"from 2022, {val['class_id'][in_2022].nunique()} classes), {len(calibration)} calibration; {DEVICE}, "
      f"{WORKERS} workers")


def save(baseline_id, metrics, outputs=None, frame=val, extra_versions=None):
    saved = save_baseline(OUT / baseline_id, metrics, {**EXTRA_VERSIONS, **(extra_versions or {})},
                          frame if outputs else None, outputs)
    print({key: value for key, value in summary_row(baseline_id, saved).items() if value == value})
    return saved

code /kaggle/working/repo/src; 13611 training photos after caps, 17151 validation (13947 uploaded from 2022, 174 classes), 7999 calibration; cuda, 4 workers


## Few-shot code (C1 and K1)

A class prototype is the normalised mean of embeddings from distinct observations, scored on the class's other
observations; a zero prototype row scores -inf, so it never wins. C1 draws prototypes for the tier 2 classes from a
frozen or trained encoder's features; K1 enrols the 10 held-out species into the K1 run's own prototypes. Both
run on the CPU from saved embeddings, so `RUN_BASELINES = []` with `K1_RUN_DIR` set re-runs K1 alone.

In [11]:
import numpy as np

from serpentguard.fewshot import masked_cosine, unit

ENROL_SHOTS, CURVE_SHOTS = (1, 5, 10), (5, 10, 20)
DRAWS, SEED = 5, 42
MIN_QUERY_OBSERVATIONS = 1  # observations a class keeps back for scoring after the largest draw
ENROL_KEYS = ("new_top1", "new_top1_other_photographer", "new_as_old")


def draw_distinct_observations(rng, obs_ids, k):
    """Positions of k photos from k different observations: k observations at random, then one photo of each."""
    obs_ids = np.asarray(obs_ids)
    chosen = rng.choice(np.unique(obs_ids), k, replace=False)
    return np.array([rng.choice(np.flatnonzero(obs_ids == obs)) for obs in chosen], dtype=int)


def nanmean(values):
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]
    return float(values.mean()) if len(values) else float("nan")


def accuracy(predicted, truth):
    return float(np.mean(np.asarray(predicted) == truth)) if len(predicted) else float("nan")


def spread(per_draw, keys):
    """Mean and sample std across draws of each key, ignoring nan draws (std 0 with fewer than two)."""
    out = {}
    for key in keys:
        values = np.array([draw[key] for draw in per_draw], dtype=float)
        values = values[~np.isnan(values)]
        out[key] = nanmean(values)
        out[f"{key}_std"] = float(values.std(ddof=1)) if len(values) > 1 else 0.0
    return out


def enrolment_eval(heldout, prototypes_old, val_old, shots=ENROL_SHOTS, draws=DRAWS, seed=SEED):
    """K1: per k and draw, enrol every held-out species from k distinct observations into its zero prototype row,
    then score its other observations and the old classes' val_old top-1 against all prototypes."""
    embedding, labels = unit(heldout["embedding"]), np.asarray(heldout["labels"], dtype=int)
    species, obs = np.asarray(heldout["species"]).astype(str), np.asarray(heldout["obs_id"]).astype(str)
    observers = np.asarray(heldout["observer_id"]).astype(str)
    prototypes_old = np.asarray(prototypes_old, dtype=float)
    old_active = np.linalg.norm(prototypes_old, axis=1) > 0
    new_ids = np.unique(labels)
    assert not old_active[new_ids].any(), "held-out classes already have prototypes"
    val_embedding, val_labels = unit(val_old["embedding"]), np.asarray(val_old["labels"], dtype=int)
    names = {int(class_id): str(species[labels == class_id][0]) for class_id in new_ids}

    before = accuracy(masked_cosine(val_embedding, prototypes_old).argmax(axis=1), val_labels)
    result = {"old_top1_before": before, "held_out_species": sorted(names.values()), "photos": len(labels),
              "draws": draws, "seed": seed, "shots": {}}
    for k in shots:
        per_draw, per_species, picks = [], {}, []
        for draw in range(draws):
            rng = np.random.default_rng([seed, k, draw])
            prototypes, enrolled = prototypes_old.copy(), {}
            for class_id in new_ids:
                members = np.flatnonzero(labels == class_id)
                if np.unique(obs[members]).size >= k + MIN_QUERY_OBSERVATIONS:
                    enrolled[int(class_id)] = members[draw_distinct_observations(rng, obs[members], k)]
                    prototypes[class_id] = unit(embedding[enrolled[int(class_id)]].mean(axis=0))[0]
            predicted = masked_cosine(embedding, prototypes).argmax(axis=1)
            scores = {}
            for class_id, picked in enrolled.items():
                members = np.flatnonzero(labels == class_id)
                queries = members[~np.isin(obs[members], obs[picked])]
                strangers = queries[~np.isin(observers[queries], observers[picked])]
                scores[names[class_id]] = {
                    "new_top1": accuracy(predicted[queries], class_id),
                    "new_top1_other_photographer": accuracy(predicted[strangers], class_id),
                    "new_as_old": float(old_active[predicted[queries]].mean()),
                    "queries": len(queries), "queries_other_photographer": len(strangers)}
            old_predicted = masked_cosine(val_embedding, prototypes).argmax(axis=1)
            per_draw.append({"draw": draw, **{key: nanmean([s[key] for s in scores.values()]) for key in ENROL_KEYS},
                             "old_top1_after": accuracy(old_predicted, val_labels),
                             "old_as_new": float(np.isin(old_predicted, list(enrolled)).mean())})
            for name, score in scores.items():
                per_species.setdefault(name, []).append(score)
            picks.append({names[class_id]: picked.tolist() for class_id, picked in enrolled.items()})
        summary = spread(per_draw, (*ENROL_KEYS, "old_top1_after", "old_as_new"))
        result["shots"][k] = {
            **summary, "old_top1_drop": before - summary["old_top1_after"], "species_enrolled": len(per_species),
            "skipped_species": sorted(set(names.values()) - set(per_species)),
            "species": {name: {key: nanmean([draw[key] for draw in draws]) for key in draws[0]}
                        for name, draws in sorted(per_species.items())},
            "per_draw": per_draw, "enrolled_photos": picks}
    return result


def fewshot_curve(features, labels, obs_ids, classes, shots=CURVE_SHOTS, draws=DRAWS, seed=SEED):
    """C1: pooled and per-class top-1 of prototypes from k distinct observations, per k, over the given classes
    that have max(shots) + MIN_QUERY_OBSERVATIONS observations, so every k scores the same classes."""
    features, labels, obs_ids = unit(features), np.asarray(labels, dtype=int), np.asarray(obs_ids).astype(str)
    needed = max(shots) + MIN_QUERY_OBSERVATIONS
    asked = sorted(int(class_id) for class_id in set(classes))
    eligible = [class_id for class_id in asked if np.unique(obs_ids[labels == class_id]).size >= needed]
    curve = {"classes": eligible, "skipped_classes": sorted(set(asked) - set(eligible)), "min_observations": needed,
             "draws": draws, "seed": seed, "shots": {}}
    for k in shots:
        per_draw = []
        for draw in range(draws):
            rng = np.random.default_rng([seed, k, draw])
            prototypes, queries = np.zeros((len(eligible), features.shape[1])), []
            for row, class_id in enumerate(eligible):
                members = np.flatnonzero(labels == class_id)
                picked = members[draw_distinct_observations(rng, obs_ids[members], k)]
                prototypes[row] = unit(features[picked].mean(axis=0))[0]
                queries.append(members[~np.isin(obs_ids[members], obs_ids[picked])])
            queries = np.concatenate(queries)
            truth = np.searchsorted(eligible, labels[queries])
            correct = masked_cosine(features[queries], prototypes).argmax(axis=1) == truth
            per_draw.append({"draw": draw, "top1": float(correct.mean()), "queries": len(queries),
                             "macro_top1": float(np.mean([correct[truth == row].mean()
                                                          for row in range(len(eligible))]))})
        curve["shots"][k] = {**spread(per_draw, ("top1", "macro_top1")),
                             "queries": float(np.mean([draw["queries"] for draw in per_draw])), "per_draw": per_draw}
    return curve


def observation_ids(paths, metadata):
    """obs_id of each path, from metadata.csv; the saved run outputs carry paths but no obs_id."""
    return metadata.set_index("path")["obs_id"].loc[list(paths)].astype(str).to_numpy()

## B1 download

The model and its label list come from the LINDAT item `20.500.12800/1-4773` (BSD-3). We read the file URLs and MD5s
from the item page saved on 22 Oct 2026, and the download checks each MD5. The labels are the SnakeCLEF 2021
training metadata's `class_id` to `binomial` pairs. The graph's name says it preprocesses its own input, so the cell
prints the model's input, output and first operations; check them before trusting B1.

In [12]:
LINDAT = "https://lindat.mff.cuni.cz/repository/server/api/core/bitstreams/handle/20.500.12800/1-4773"
BOLON_MODEL_URL = f"{LINDAT}/ViT_PreProcessing-ops11-preprocessing-int-dynam_graph.onnx?sequence=7&isAllowed=y"
BOLON_MODEL_MD5 = "838ae0f8ed19ffeee89aba5fa8b50956"
BOLON_LABELS_URL = f"{LINDAT}/min-train_metadata.csv?sequence=2&isAllowed=y"
BOLON_LABELS_MD5 = "52ec43c8595a3835da6aa16364aff75e"
BOLON_INPUT_SIZE = None        # used only if the graph doesn't resize its input itself: then set it from the printout
BOLON_MEAN = BOLON_STD = None  # 0-255 pixels go in; set them only for a float input the graph doesn't normalise
BOLON_CHANNEL_ORDER = "RGB"    # the training code converts cv2's BGR to RGB before the model (BOLON_DOCUMENTATION)
BOLON_DOCUMENTATION = (
    "github.com/picekl/PlosNTD-GlobalSnakeID, README and notebooks/SnakeCLEF2021-ViT_large_patch16_384.ipynb, read "
    "5 Nov 2026: ViT-Large/16 at 384 x 384 with N_CLASSES = 772; the validation transform is Resize(384, 384), a "
    "squash, then Normalize with timm's default mean and std, on photos read with cv2 and converted BGR -> RGB; the "
    "training target is the metadata's class_id. The LINDAT item 20.500.12800/1-4773 documents only the dataset.")
GRAPH_OPS_RECORDED = 60
DOWNLOAD_TIMEOUT_SECONDS, CHUNK_BYTES = 120, 2 ** 24
LABEL_SAMPLE = 8


def download(url, path, md5):
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        partial = path.with_name(path.name + ".part")
        with urllib.request.urlopen(url, timeout=DOWNLOAD_TIMEOUT_SECONDS) as response, open(partial, "wb") as out:
            shutil.copyfileobj(response, out, CHUNK_BYTES)
        partial.rename(path)
    digest = hashlib.md5()
    with open(path, "rb") as stored:
        for chunk in iter(lambda: stored.read(CHUNK_BYTES), b""):
            digest.update(chunk)
    assert digest.hexdigest() == md5, f"{path.name}: MD5 {digest.hexdigest()}, expected {md5}"
    return path


if "B1" in RUN_BASELINES:
    bolon_path = download(BOLON_MODEL_URL, DOWNLOADS / "bolon_vit.onnx", BOLON_MODEL_MD5)
    bolon_session = open_onnx(bolon_path)
    for spec in bolon_session.get_inputs() + bolon_session.get_outputs():
        print(f"{spec.name}: shape {spec.shape}, {spec.type}")
    bolon_outputs = bolon_session.get_outputs()[0].shape[-1]
    assert isinstance(bolon_outputs, int), f"the model's output width is {bolon_outputs!r}, not a fixed number"
    bolon_labels = labels_from_metadata(pd.read_csv(  # an id with no photo in the metadata leaves a None slot
        download(BOLON_LABELS_URL, DOWNLOADS / "snakeclef2021_min_train_metadata.csv", BOLON_LABELS_MD5),
        low_memory=False), bolon_outputs)
    bolon_unnamed = sum(name is None for name in bolon_labels)
    print(f"providers {bolon_session.get_providers()}; {bolon_outputs} outputs, {bolon_unnamed} without a name in "
          f"the metadata; e.g. {[name for name in bolon_labels if name is not None][:LABEL_SAMPLE]}")
    bolon_graph = describe_onnx_graph(bolon_path)  # how the graph treats its input decides how we feed it
    print(f"{bolon_graph['nodes']} nodes; resizes its input itself: {bolon_graph['resizes']}; ops before the first "
          f"conv: {bolon_graph['ops_before_first_conv'][:GRAPH_OPS_RECORDED]}")
    for node in bolon_graph["first_nodes"]:
        print(f"  {node['op']}: {node['inputs']} -> {node['outputs']} {node['attributes']}")
    print(f"small constants read by those nodes: {bolon_graph['small_constants']}")
    if bolon_graph["resizes"]:
        bolon_input_size = None  # each photo goes in at its own size, after load_rgb, as the graph's dtype
    else:
        assert BOLON_INPUT_SIZE is not None, ("the graph doesn't resize its input: set BOLON_INPUT_SIZE from the "
                                              "printout")
        bolon_input_size = BOLON_INPUT_SIZE
    bolon = OnnxClassifier(bolon_path, bolon_labels, bolon_input_size, BOLON_MEAN, BOLON_STD, session=bolon_session,
                           channel_order=BOLON_CHANNEL_ORDER)
    bolon_preprocessing = {
        "choice": bolon.describe()["feed"], "channel_order": BOLON_CHANNEL_ORDER, "documentation": BOLON_DOCUMENTATION,
        "graph": {"inputs": bolon_graph["inputs"], "outputs": bolon_graph["outputs"],
                  "resizes": bolon_graph["resizes"],
                  "ops_before_first_conv": bolon_graph["ops_before_first_conv"][:GRAPH_OPS_RECORDED],
                  "first_ops": [node["op"] for node in bolon_graph["first_nodes"]],
                  "small_constants": bolon_graph["small_constants"]}}
    print(f"feed: {bolon_preprocessing['choice']}, {BOLON_CHANNEL_ORDER}")
    _, label_coverage = map_labels(bolon_labels, taxa.species)
    print(f"input {bolon.describe()['layout']} {bolon.input_type}; covers {label_coverage['classes_covered']} of our "
          f"{label_coverage['classes']} classes; renamed {label_coverage['renamed']}")

input: shape ['?', '?', 3], tensor(int32)
output: shape [1, 772], tensor(float)
2404: shape [1, 772], tensor(float)
providers ['CUDAExecutionProvider', 'CPUExecutionProvider']; 772 outputs, 4 without a name in the metadata; e.g. ['Acanthophis antarcticus', 'Acanthophis hawkei', 'Acanthophis praelongus', 'Acanthophis pyrrhus', 'Acanthophis rugosus', 'Acanthophis wellsi', 'Achalinus formosanus', 'Achalinus niger']
1909 nodes; resizes its input itself: True; ops before the first conv: ['Cast', 'Transpose', 'Unsqueeze', 'Constant', 'Div', 'Shape', 'Constant', 'Constant', 'Constant', 'Slice', 'Concat', 'Constant', 'Constant', 'Resize', 'Constant', 'Sub', 'Constant', 'Div', 'Cast']
  Cast: ['input'] -> ['297'] {'to': 1}
  Transpose: ['297'] -> ['298'] {'perm': [2, 0, 1]}
  Unsqueeze: ['298'] -> ['299'] {'axes': [0]}
  Constant: [] -> ['300'] {}
  Div: ['299', '300'] -> ['301'] {}
  Shape: ['301'] -> ['303'] {}
  Constant: [] -> ['304'] {}
  Constant: [] -> ['305'] {}
  Constant: [] -> ['306'

In [13]:
if "B0" in RUN_BASELINES:
    probs = prior_baseline(train_counts, len(val))
    save("B0", baseline_metrics(val, probs, train_counts, taxa, in_2022,
                                {**RUN, "run_id": "B0", "baseline": "class frequencies of the capped subset"}))

{'model': 'B0', 'evaluation': 'full_validation', 'macro_f1': 0.0010958070676513833, 'top1': 0.09451343944959478, 'top3': 0.21829630925310478, 'who_recall': 0.0, 'o3_raw': 1.0, 'ece': 0.021043451939489116, 'macro_f1_2022': 0.0010662587457354897, 'top1_2022': 0.09105900910590091, 'slice_overlap': 'no labelled photos from outside our training subset'}


In [14]:
B1_CHECK_PHOTOS = 256
CHANNEL_ORDER_MARGIN = 0.05  # the reversed channel order may not beat BOLON_CHANNEL_ORDER by more than this

if "B1" in RUN_BASELINES:
    covered = val[val["class_id"].isin(label_coverage["covered_class_ids"])]
    covered = covered.sample(min(B1_CHECK_PHOTOS, len(covered)), random_state=42)
    check_probs, _ = bolon_baseline(bolon, covered, taxa.species, batch_size=BATCH, num_workers=WORKERS)
    b1_check = check_label_order(check_probs, covered["class_id"].to_numpy(), label_coverage["covered_class_ids"])
    reversed_order = "BGR" if BOLON_CHANNEL_ORDER == "RGB" else "RGB"
    reversed_probs, _ = bolon_baseline(
        OnnxClassifier(bolon_path, bolon_labels, bolon_input_size, BOLON_MEAN, BOLON_STD, session=bolon_session,
                       channel_order=reversed_order), covered, taxa.species, batch_size=BATCH, num_workers=WORKERS)
    reversed_top1 = float((reversed_probs.argmax(axis=1) == covered["class_id"].to_numpy()).mean())
    bolon_preprocessing["channel_probe"] = {BOLON_CHANNEL_ORDER: b1_check["top1"], reversed_order: reversed_top1}
    print(f"B1 top-1 on {b1_check['photos']} validation photos of covered classes: {b1_check['top1']:.3f} "
          f"(needs {b1_check['threshold']:.3f}; chance {b1_check['chance']:.4f}); with {reversed_order} channels "
          f"{reversed_top1:.3f}")
    assert reversed_top1 <= b1_check["top1"] + CHANNEL_ORDER_MARGIN, (
        f"{reversed_order} channels score {reversed_top1:.3f} against {b1_check['top1']:.3f}: check "
        f"BOLON_CHANNEL_ORDER against the graph printout")

B1 top-1 on 256 validation photos of covered classes: 0.820 (needs 0.161; chance 0.0161); with BGR channels 0.613


In [15]:
B1_PROGRESS_EVERY = 1000  # one photo per run on a ViT-L/16 at 384: about 15-25 photos per second on a T4

if "B1" in RUN_BASELINES:
    started = time.perf_counter()
    probs, coverage = bolon_baseline(bolon, val, taxa.species, batch_size=BATCH, num_workers=WORKERS,
                                     progress_every=B1_PROGRESS_EVERY)
    coverage["photo_share_upload_2022_slice"] = photo_share(val["class_id"][in_2022], coverage["covered_class_ids"])
    run = {**RUN, "run_id": "B1", "baseline": "Bolon et al. 2022 global ViT (ONNX, LINDAT 20.500.12800/1-4773)",
           "onnx": bolon.describe(), "preprocessing": bolon_preprocessing, "model_url": BOLON_MODEL_URL,
           "model_md5": BOLON_MODEL_MD5,
           "labels_url": BOLON_LABELS_URL, "labels_md5": BOLON_LABELS_MD5,
           "label_columns": {"id": BOLON_ID_COLUMN, "name": BOLON_NAME_COLUMN}, "synonyms": SYNONYMS,
           "unnamed_outputs": bolon_unnamed, "label_order_check": b1_check,
           "seconds": time.perf_counter() - started}
    save("B1", baseline_metrics(val, probs, train_counts, taxa, in_2022, run, evaluation=UPPER_BOUND,
                                slice_overlap=OVERLAP_CLEARED, extra={"coverage": coverage}),
         {"probs": probs.astype(np.float32)}, extra_versions={"bolon_model_md5": BOLON_MODEL_MD5})

1000/17151 photos, 8.4 per second
2000/17151 photos, 8.1 per second
3000/17151 photos, 8.0 per second
4000/17151 photos, 8.0 per second
5000/17151 photos, 8.0 per second
6000/17151 photos, 7.9 per second
7000/17151 photos, 7.9 per second
8000/17151 photos, 7.9 per second
9000/17151 photos, 7.9 per second
10000/17151 photos, 7.9 per second
11000/17151 photos, 7.9 per second
12000/17151 photos, 7.9 per second
13000/17151 photos, 7.9 per second
14000/17151 photos, 7.9 per second
15000/17151 photos, 7.9 per second
16000/17151 photos, 7.9 per second
17000/17151 photos, 7.9 per second
17151/17151 photos, 7.9 per second
{'model': 'B1', 'evaluation': 'upper_bound_overlap_unverified', 'macro_f1': 0.2568058920073597, 'top1': 0.6825258002448836, 'top3': 0.7706256194974054, 'who_recall': 0.7780384795870483, 'o3_raw': 0.22196152041295167, 'ece': 0.14584648418496474, 'macro_f1_2022': 0.2521075385221927, 'top1_2022': 0.6741234674123467, 'slice_overlap': "clear by date: the model's training photos pre

In [16]:
if "B2" in RUN_BASELINES:
    started = time.perf_counter()
    bioclip, _, bioclip_preprocess = open_clip.create_model_and_transforms(BIOCLIP_HUB)
    class_names = taxonomic_names(tables.classes)
    print(f"prompts like {BIOCLIP_TEMPLATES[0].format(class_names[0])!r}")
    probs = bioclip_baseline(bioclip.to(DEVICE), open_clip.get_tokenizer(BIOCLIP_HUB), bioclip_preprocess, val,
                             class_names, BIOCLIP_TEMPLATES, batch_size=BATCH, num_workers=WORKERS, device=DEVICE)
    run = {**RUN, "run_id": "B2", "baseline": f"BioCLIP zero-shot ({BIOCLIP_HUB}), taxonomic prompts",
           "hub": BIOCLIP_HUB, "templates": list(BIOCLIP_TEMPLATES), "lineage": BIOCLIP_LINEAGE,
           "logit_scale": logit_scale(bioclip), "preprocess": repr(bioclip_preprocess),
           "seconds": time.perf_counter() - started}
    save("B2", baseline_metrics(val, probs, train_counts, taxa, in_2022, run, evaluation=UPPER_BOUND,
                                slice_overlap=OVERLAP_NOT_VERIFIED), {"probs": probs.astype(np.float32)})
    del bioclip
    torch.cuda.empty_cache()

open_clip_pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  599MB            

open_clip_pytorch_model.bin: downloading bytes:           |  0.00B            

open_clip_config.json:   0%|          | 0.00/469 [00:00<?, ?B/s]

prompts like 'a photo of Animalia Chordata Reptilia Squamata Typhlopidae Indotyphlops braminus.'
{'model': 'B2', 'evaluation': 'upper_bound_overlap_unverified', 'macro_f1': 0.1867267431642731, 'top1': 0.3542650574310536, 'top3': 0.5252171885021282, 'who_recall': 0.4373533552322853, 'o3_raw': 0.5626466447677146, 'ece': 0.31062396540697046, 'macro_f1_2022': 0.17338912733399958, 'top1_2022': 0.3397146339714634, 'slice_overlap': "overlap not verified: the model's training data runs past 2022"}


In [17]:
if "B3" in RUN_BASELINES:
    for encoder_name, suffix in ENCODER_IDS.items():
        started = time.perf_counter()
        encoder = frozen_encoder(encoder_name)
        features_train, features_val = (embed_frozen(encoder_name, frame, batch_size=BATCH, num_workers=WORKERS,
                                                     device=DEVICE, encoder=encoder) for frame in (train, val))
        probs = frozen_prototype_baseline(features_train, train["class_id"], features_val, n_classes=taxa.n_classes)
        run = {**RUN, "run_id": f"B3_{suffix}", "baseline": f"{encoder_name}, frozen ImageNet weights",
               "head_is": "nearest prototype from the subset, softmax(prototype_scale x cosine)",
               **frozen_run_record(encoder_name, encoder, INPUT_SIZE), "seconds": time.perf_counter() - started}
        save(f"B3_{suffix}", baseline_metrics(val, probs, train_counts, taxa, in_2022, run, prototype_probs=probs),
             {"probs": probs.astype(np.float32)})

model.safetensors: reconstructing file:   0%|          |  0.00B / 15.2MB            

model.safetensors: downloading bytes:           |  0.00B            

{'model': 'B3_mnv4', 'evaluation': 'full_validation', 'macro_f1': 0.09239042613103067, 'macro_f1_prototype': 0.09239042613103067, 'top1': 0.156608944084893, 'top3': 0.2978835053349659, 'who_recall': 0.328484279680901, 'o3_raw': 0.671515720319099, 'ece': 0.037257473830600844, 'macro_f1_2022': 0.09065020996875518, 'top1_2022': 0.15551731555173157, 'slice_overlap': 'no labelled photos from outside our training subset'}


model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

{'model': 'B3_effb0', 'evaluation': 'full_validation', 'macro_f1': 0.14001803996856246, 'macro_f1_prototype': 0.14001803996856246, 'top1': 0.2287330184828873, 'top3': 0.40767302198122557, 'who_recall': 0.4514312529328954, 'o3_raw': 0.5485687470671047, 'ece': 0.039028885938134014, 'macro_f1_2022': 0.13524535755923214, 'top1_2022': 0.22528142252814226, 'slice_overlap': 'no labelled photos from outside our training subset'}


model.safetensors: reconstructing file:   0%|          |  0.00B / 22.4MB            

model.safetensors: downloading bytes:           |  0.00B            

{'model': 'B3_mvits', 'evaluation': 'full_validation', 'macro_f1': 0.13520903996764572, 'macro_f1_prototype': 0.13520903996764572, 'top1': 0.2171885021281558, 'top3': 0.4001515946592035, 'who_recall': 0.44251525105584233, 'o3_raw': 0.5574847489441577, 'ece': 0.03226574227012736, 'macro_f1_2022': 0.1319647582534319, 'top1_2022': 0.21488492148849214, 'slice_overlap': 'no labelled photos from outside our training subset'}


In [18]:
if "B4" in RUN_BASELINES or "C1" in RUN_BASELINES:
    started = time.perf_counter()
    dino_encoder = frozen_encoder(DINOV2, DINOV2_INPUT_SIZE)
    dino = {name: embed_frozen(DINOV2, frame, input_size=DINOV2_INPUT_SIZE, batch_size=BATCH, num_workers=WORKERS,
                               device=DEVICE, encoder=dino_encoder)
            for name, frame in (("train", train), ("val", val), ("calibration", calibration))}
    embed_seconds = time.perf_counter() - started
if "B4" in RUN_BASELINES:
    probe = linear_probe(dino["train"], train["class_id"], dino["val"], n_classes=taxa.n_classes)
    prototype = frozen_prototype_baseline(dino["train"], train["class_id"], dino["val"], n_classes=taxa.n_classes)
    run = {**RUN, "run_id": "B4", "baseline": f"{DINOV2} frozen at {DINOV2_INPUT_SIZE} px",
           "head_is": "linear probe: multinomial logistic regression (lbfgs) on L2-normalised features",
           **frozen_run_record(DINOV2, dino_encoder, DINOV2_INPUT_SIZE), "probe_c": PROBE_C,
           "probe_max_iter": PROBE_MAX_ITER, "probe_class_weight": "balanced", "embed_seconds": embed_seconds,
           "seconds": time.perf_counter() - started}
    save("B4", baseline_metrics(val, probe, train_counts, taxa, in_2022, run, prototype_probs=prototype),
         {"probs": probe.astype(np.float32), "prototype_probs": prototype.astype(np.float32)})
    np.savez_compressed(OUT / "B4" / "features.npz", **{f"{name}_features": value for name, value in dino.items()},
                        **{f"{name}_paths": frame["path"].to_numpy(dtype=str) for name, frame in
                           (("train", train), ("val", val), ("calibration", calibration))})

model.safetensors: reconstructing file:   0%|          |  0.00B / 88.2MB            

model.safetensors: downloading bytes:           |  0.00B            

{'model': 'B4', 'evaluation': 'full_validation', 'macro_f1': 0.10513238744121663, 'macro_f1_prototype': 0.07775157190626454, 'top1': 0.1801061162614425, 'top3': 0.34085476065535536, 'who_recall': 0.35335523228531207, 'o3_raw': 0.6466447677146879, 'ece': 0.11111744329689408, 'macro_f1_2022': 0.10213595334978706, 'top1_2022': 0.1811859181185918, 'slice_overlap': 'no labelled photos from outside our training subset'}


In [19]:
def tier2_curve(features, frame, description):
    curve = fewshot_curve(features, frame["class_id"], observation_ids(frame["path"], tables.metadata),
                          np.flatnonzero(taxa.tier == TIER2))
    curve["features"], curve["pool"] = description, "validation and calibration photos (never trained on)"
    for shots, scores in curve["shots"].items():
        print(f"{description}, {shots} shots: top-1 {scores['top1']:.3f} +/- {scores['top1_std']:.3f}, "
              f"per-class {scores['macro_top1']:.3f} on {len(curve['classes'])} tier 2 classes")
    return curve


if "C1" in RUN_BASELINES:
    pool = pd.concat([val, calibration], ignore_index=True)
    write_json(FEWSHOT_OUT / "C1_dinov2.json",
               tier2_curve(np.concatenate([dino["val"], dino["calibration"]]), pool,
                           f"{DINOV2} frozen, {DINOV2_INPUT_SIZE} px"))

vit_small_patch14_dinov2 frozen, 322 px, 5 shots: top-1 0.238 +/- 0.020, per-class 0.221 on 28 tier 2 classes
vit_small_patch14_dinov2 frozen, 322 px, 10 shots: top-1 0.301 +/- 0.010, per-class 0.276 on 28 tier 2 classes
vit_small_patch14_dinov2 frozen, 322 px, 20 shots: top-1 0.359 +/- 0.026, per-class 0.312 on 28 tier 2 classes


In [20]:
if "B5" in RUN_BASELINES:
    from ultralytics import YOLO

    started = time.perf_counter()
    detector = YOLO(YOLO_WEIGHTS)
    images, truth = open_images_ground_truth(pd.read_csv(EXTENDED / "open_images_snake_boxes.csv"),
                                             pd.read_csv(EXTENDED / "open_images_snake_masks.csv"))
    if LIMIT is not None:
        images, truth = images.head(LIMIT), truth[:LIMIT]
    boxes = yolo_snake_boxes(detector, [str(EXTENDED / name) for name in images["image_file"]], normalized=True)
    run = {**RUN, "run_id": "B5", "baseline": f"{YOLO_WEIGHTS} (Open Images V7), as is",
           "evaluation": OPEN_IMAGES_EVALUATION, "weights": YOLO_WEIGHTS,
           "snake_class": snake_class_index(detector.names),
           "conf": YOLO_CONF, "all_conf": YOLO_ALL_CONF, "iou_threshold": DET_IOU, "batch": YOLO_BATCH,
           "images": len(images), "rotated_images": int(images["rotation"].fillna(0).ne(0).sum()),
           "boxes": "normalised to the photo; IoU doesn't change when each axis is scaled",
           "seconds": time.perf_counter() - started}
    save("B5", {"detection": detection_baseline_scores(boxes, truth), "run": run})
    write_json(OUT / "B5" / "predictions.json", {"image_id": images["image_id"].tolist(), "boxes": boxes})

{'model': 'B5', 'evaluation': 'open_images_snake_boxes', 'precision': 0.8227848101265823, 'recall': 0.2181208053691275, 'map50': 0.510067678677651}


## B6 is not run

B6 would score Grounding DINO + SAM 2.1, our M2 teacher, on the Open Images masks. We left it out of M3 as optional:
it needs notebook 04's stack for 284 photos, and M2 already reports the teacher's quality by eye (84% good of 491). It can be added in M4.

In [21]:
if E2_RUN_DIR is not None:
    parts = [np.load(Path(E2_RUN_DIR) / name) for name in ("val_outputs.npz", "calib_outputs.npz")]
    pool = pd.DataFrame({"path": np.concatenate([part["paths"] for part in parts]),
                         "class_id": np.concatenate([part["labels"] for part in parts])})
    write_json(FEWSHOT_OUT / "C1_E2.json", tier2_curve(np.concatenate([part["embedding"] for part in parts]), pool,
                                                       f"E2 embeddings from {Path(E2_RUN_DIR).name}"))
else:
    print("E2_RUN_DIR is None: C1 on E2 features waits for an E2 run")

E2_RUN_DIR is None: C1 on E2 features waits for an E2 run


In [22]:
PROTOTYPE_KINDS = ("plain", "balanced")

if K1_RUN_DIR is not None:
    run_dir = Path(K1_RUN_DIR)
    with np.load(run_dir / "heldout_outputs.npz") as heldout, np.load(run_dir / "prototypes.npz") as prototypes, \
            np.load(run_dir / "val_outputs.npz") as val_old:
        heldout, val_old = dict(heldout), dict(val_old)
        enrolment = {kind: enrolment_eval(heldout, prototypes[kind], val_old) for kind in PROTOTYPE_KINDS}
    write_json(FEWSHOT_OUT / "K1_enrolment.json", {"run": run_dir.name, **enrolment})
    for kind, result in enrolment.items():
        for shots, scores in result["shots"].items():
            print(f"K1 {kind}, {shots} shots: new top-1 {scores['new_top1']:.3f} +/- {scores['new_top1_std']:.3f} "
                  f"(other photographers {scores['new_top1_other_photographer']:.3f}), called an old class "
                  f"{scores['new_as_old']:.3f}; old top-1 {result['old_top1_before']:.3f} -> "
                  f"{scores['old_top1_after']:.3f}")
else:
    print("K1_RUN_DIR is None: K1 enrolment waits for the K1 run")

K1_RUN_DIR is None: K1 enrolment waits for the K1 run


In [23]:
rows = [summary_row(path.parent.name, json.loads(path.read_text()))
        for path in sorted(OUT.glob("*/final_metrics.json"))]
comparison = pd.DataFrame(rows).set_index("model") if rows else pd.DataFrame()
comparison.to_csv(OUT / "comparison.csv")
missing = [baseline for baseline in RUN_BASELINES if baseline != "C1" and not any(
    row["model"].startswith(baseline) for row in rows)]
assert not missing, f"baselines without a final_metrics.json: {missing}"
print(comparison.round(3).to_string())

                              evaluation  macro_f1  macro_f1_prototype   top1   top3  who_recall  o3_raw    ece  macro_f1_2022  top1_2022                                                   slice_overlap  class_coverage  photo_coverage  precision  recall  map50
model                                                                                                                                                                                                                                                              
B0                       full_validation     0.001                 NaN  0.095  0.218       0.000   1.000  0.021          0.001      0.091             no labelled photos from outside our training subset             NaN             NaN        NaN     NaN    NaN
B1        upper_bound_overlap_unverified     0.257                 NaN  0.683  0.771       0.778   0.222  0.146          0.252      0.674         clear by date: the model's training photos predate 2022           0.337   